In [1]:
import pandas as pd
import numpy as np
from plotly import tools
from plotly.offline import iplot
import plotly.graph_objects as go
import warnings
warnings.simplefilter('ignore')
import matplotlib.cm as cm
import matplotlib.colors as mcolors
lsls=[]
lstb=[]

def length(x):
    df=pd.DataFrame(x,columns=['x','y','z'])
    disc_dst=np.sqrt((df['x'].diff()**2) + (df['y'].diff()**2) + (df['z'].diff()**2))
    lngth=disc_dst.sum()
    return lngth

def plot_curvature(siml,color,file_name):
    def curvature(df):
        x = df['X'].values
        y = df['Y'].values
        z = df['Z'].values
        dx = np.gradient(x)
        dy = np.gradient(y)
        dz = np.gradient(z)
        ddx = np.gradient(dx)
        ddy = np.gradient(dy)
        ddz = np.gradient(dz)
        cross = np.cross(np.vstack((dx,dy,dz)).T, np.vstack((ddx,ddy,ddz)).T)
        cross_norm = np.linalg.norm(cross, axis=1)
        first_norm = np.linalg.norm(np.vstack((dx,dy,dz)).T, axis=1)
        curvature = cross_norm / (first_norm**3)
        return curvature
    fig2=go.Scatter(x=[i for i in range(1,101,1)], y=curvature(siml),mode='lines',name=file_name,line=dict(color=color,width=3))
        # fig3=go.Scatter(x=[i for i in range(1,101,1)], y=curvature(prt),mode='lines',name='Prototype',line=dict(color='dodgerblue',width=3))
    return fig2

In [2]:
#| label: w1-main

df=pd.read_csv('../../data/2001/w1.csv')
doe_df=pd.read_csv('../../pyelastica/wiper_doe10p/data/doe_df.csv')[0:100]
prt=df.iloc[10:,2:].reset_index(drop=True).values
p0=df.iloc[5,2:].values.astype(float)   
p1=df.iloc[6,2:].values.astype(float)   
u0=df.iloc[8,2:].values.astype(float)*df.iloc[2,1]   
u1=df.iloc[9,2:].values.astype(float)*df.iloc[3,1]   
fig = go.Figure()
fig.add_trace(go.Scatter3d(x=prt[:,0],y=prt[:,1],z=prt[:,2],mode='lines',name='Prototype',line=dict(color='black',width=6)))
fig.add_trace(go.Scatter3d(x=[p0[0]],y=[p0[1]],z=[p0[2]],mode='markers',name='P0',marker=dict(size=5,color='red')))
fig.add_trace(go.Scatter3d(x=[p1[0]],y=[p1[1]],z=[p1[2]],mode='markers',name='P1',marker=dict(size=5,color='red')))
fig.add_trace(go.Cone(x=[p0[0]],y=[p0[1]],z=[p0[2]],u=[u0[0]],v=[u0[1]],w=[u0[2]],colorscale='RdPu',sizemode='absolute',sizeref=6,showscale=False,name='U@P0',showlegend=True))
fig.add_trace(go.Cone(x=[p1[0]],y=[p1[1]],z=[p1[2]],u=[u1[0]],v=[u1[1]],w=[u1[2]],colorscale='RdPu',sizemode='absolute',sizeref=6,showscale=False,name='U@P1',showlegend=True))
a=pd.DataFrame(prt).iloc[50].values

lst_err=[]
lst_ln=[]

angles = list(range(0,20))
cmap = cm.get_cmap('hsv', len(angles))   # choose any colormap

for idx, i in enumerate(angles):
    file_name=f'N={i+1}'
    spl3=pd.read_csv('../../pyelastica/wiper_doe10p/data/w1_iter_'+str(i+1)+'.csv')
    errb=length(pd.DataFrame([a,spl3.iloc[50]]).values).round(3)
    spl3=spl3.values
    color = mcolors.to_hex(cmap(idx))
    fig.add_trace(go.Scatter3d(x=spl3[:,0],y=spl3[:,1],z=spl3[:,2],mode='lines',name=file_name,line=dict(color=color,width=6),customdata=np.tile(np.array([i,doe_df["twist"].iloc[i],doe_df["youngs_modulus"].iloc[i]*1000,doe_df['base_radius'].iloc[i],doe_df['density'].iloc[i]*10e5,doe_df['coner_angle'].iloc[i],doe_df['length'].iloc[i]]), (spl3.shape[0],1)),hovertemplate=f"N= {i+1}<br>"+"twist= %{customdata[1]:.2f} deg<br>"+"E= %{customdata[2]:.2f} MPa<br>"+"r= %{customdata[3]:.2f} mm<br>"+"ro= %{customdata[4]:.2f} g/cm3<br>"+"cone angle= %{customdata[5]:.2f} deg<br>"+"length= %{customdata[6]:.2f} mm<br>"+"<extra></extra>"))
    ln_spl=length(spl3)-length(prt)
    lst_err.append(errb)
    lst_ln.append(ln_spl)
fig.update_layout(title='Simulated - W1 - DOE',scene=dict(xaxis_title='X',yaxis_title='Y',zaxis_title='Z',camera=dict(eye=dict(x=1,y=-1,z=1))),font=dict(size=15,family='Times New Roman'),autosize=False,height=550,width=600,margin=dict(l=20,r=20,t=40,b=20),showlegend=False) # ,legend=dict(x=0.02,y=0.98),
fig.update_scenes(camera_projection_type='orthographic')
fig.show()

In [3]:
#| label: w2-main

df=pd.read_csv('../../data/2001/w2.csv')
doe_df=pd.read_csv('../../pyelastica/wiper_doe10p/data/doe_df.csv')[0:100]
prt=df.iloc[10:,2:].reset_index(drop=True).values
p0=df.iloc[5,2:].values.astype(float)   
p1=df.iloc[6,2:].values.astype(float)   
u0=df.iloc[8,2:].values.astype(float)*df.iloc[2,1]   
u1=df.iloc[9,2:].values.astype(float)*df.iloc[3,1]   
fig = go.Figure()
fig.add_trace(go.Scatter3d(x=prt[:,0],y=prt[:,1],z=prt[:,2],mode='lines',name='Prototype',line=dict(color='black',width=6)))
fig.add_trace(go.Scatter3d(x=[p0[0]],y=[p0[1]],z=[p0[2]],mode='markers',name='P0',marker=dict(size=5,color='red')))
fig.add_trace(go.Scatter3d(x=[p1[0]],y=[p1[1]],z=[p1[2]],mode='markers',name='P1',marker=dict(size=5,color='red')))
fig.add_trace(go.Cone(x=[p0[0]],y=[p0[1]],z=[p0[2]],u=[u0[0]],v=[u0[1]],w=[u0[2]],colorscale='RdPu',sizemode='absolute',sizeref=6,showscale=False,name='U@P0',showlegend=True))
fig.add_trace(go.Cone(x=[p1[0]],y=[p1[1]],z=[p1[2]],u=[u1[0]],v=[u1[1]],w=[u1[2]],colorscale='RdPu',sizemode='absolute',sizeref=6,showscale=False,name='U@P1',showlegend=True))
a=pd.DataFrame(prt).iloc[50].values

lst_err=[]
lst_ln=[]

angles = list(range(0,20))
cmap = cm.get_cmap('hsv', len(angles))   # choose any colormap

for idx, i in enumerate(angles):
    file_name=f'N={i+1}'
    spl3=pd.read_csv('../../pyelastica/wiper_doe10p/data/w2_iter_'+str(i+1)+'.csv')
    errb=length(pd.DataFrame([a,spl3.iloc[50]]).values).round(3)
    spl3=spl3.values
    color = mcolors.to_hex(cmap(idx))
    fig.add_trace(go.Scatter3d(x=spl3[:,0],y=spl3[:,1],z=spl3[:,2],mode='lines',name=file_name,line=dict(color=color,width=6),customdata=np.tile(np.array([i,doe_df["twist"].iloc[i],doe_df["youngs_modulus"].iloc[i]*1000,doe_df['base_radius'].iloc[i],doe_df['density'].iloc[i]*10e5,doe_df['coner_angle'].iloc[i],doe_df['length'].iloc[i]]), (spl3.shape[0],1)),hovertemplate=f"N= {i+1}<br>"+"twist= %{customdata[1]:.2f} deg<br>"+"E= %{customdata[2]:.2f} MPa<br>"+"r= %{customdata[3]:.2f} mm<br>"+"ro= %{customdata[4]:.2f} g/cm3<br>"+"cone angle= %{customdata[5]:.2f} deg<br>"+"length= %{customdata[6]:.2f} mm<br>"+"<extra></extra>"))
    ln_spl=length(spl3)-length(prt)
    lst_err.append(errb)
    lst_ln.append(ln_spl)
fig.update_layout(title='Simulated - W2 - DOE',scene=dict(xaxis_title='X',yaxis_title='Y',zaxis_title='Z',camera=dict(eye=dict(x=1,y=-1,z=1))),font=dict(size=15,family='Times New Roman'),autosize=False,height=550,width=600,margin=dict(l=20,r=20,t=40,b=20),showlegend=False) # ,legend=dict(x=0.02,y=0.98),
fig.update_scenes(camera_projection_type='orthographic')
fig.show()

In [4]:
#| label: w3-main

df=pd.read_csv('../../data/2001/w3.csv')
doe_df=pd.read_csv('../../pyelastica/wiper_doe10p/data/doe_df.csv')[0:100]
prt=df.iloc[10:,2:].reset_index(drop=True).values
p0=df.iloc[5,2:].values.astype(float)   
p1=df.iloc[6,2:].values.astype(float)   
u0=df.iloc[8,2:].values.astype(float)*df.iloc[2,1]   
u1=df.iloc[9,2:].values.astype(float)*df.iloc[3,1]   
fig = go.Figure()
fig.add_trace(go.Scatter3d(x=prt[:,0],y=prt[:,1],z=prt[:,2],mode='lines',name='Prototype',line=dict(color='black',width=6)))
fig.add_trace(go.Scatter3d(x=[p0[0]],y=[p0[1]],z=[p0[2]],mode='markers',name='P0',marker=dict(size=5,color='red')))
fig.add_trace(go.Scatter3d(x=[p1[0]],y=[p1[1]],z=[p1[2]],mode='markers',name='P1',marker=dict(size=5,color='red')))
fig.add_trace(go.Cone(x=[p0[0]],y=[p0[1]],z=[p0[2]],u=[u0[0]],v=[u0[1]],w=[u0[2]],colorscale='RdPu',sizemode='absolute',sizeref=6,showscale=False,name='U@P0',showlegend=True))
fig.add_trace(go.Cone(x=[p1[0]],y=[p1[1]],z=[p1[2]],u=[u1[0]],v=[u1[1]],w=[u1[2]],colorscale='RdPu',sizemode='absolute',sizeref=6,showscale=False,name='U@P1',showlegend=True))
a=pd.DataFrame(prt).iloc[50].values

lst_err=[]
lst_ln=[]

angles = list(range(0,20))
cmap = cm.get_cmap('hsv', len(angles))   # choose any colormap

for idx, i in enumerate(angles):
    file_name=f'N={i+1}'
    spl3=pd.read_csv('../../pyelastica/wiper_doe10p/data/w3_iter_'+str(i+1)+'.csv')
    errb=length(pd.DataFrame([a,spl3.iloc[50]]).values).round(3)
    spl3=spl3.values
    color = mcolors.to_hex(cmap(idx))
    fig.add_trace(go.Scatter3d(x=spl3[:,0],y=spl3[:,1],z=spl3[:,2],mode='lines',name=file_name,line=dict(color=color,width=6),customdata=np.tile(np.array([i,doe_df["twist"].iloc[i],doe_df["youngs_modulus"].iloc[i]*1000,doe_df['base_radius'].iloc[i],doe_df['density'].iloc[i]*10e5,doe_df['coner_angle'].iloc[i],doe_df['length'].iloc[i]]), (spl3.shape[0],1)),hovertemplate=f"N= {i+1}<br>"+"twist= %{customdata[1]:.2f} deg<br>"+"E= %{customdata[2]:.2f} MPa<br>"+"r= %{customdata[3]:.2f} mm<br>"+"ro= %{customdata[4]:.2f} g/cm3<br>"+"cone angle= %{customdata[5]:.2f} deg<br>"+"length= %{customdata[6]:.2f} mm<br>"+"<extra></extra>"))
    ln_spl=length(spl3)-length(prt)
    lst_err.append(errb)
    lst_ln.append(ln_spl)
fig.update_layout(title='Simulated - W3 - DOE',scene=dict(xaxis_title='X',yaxis_title='Y',zaxis_title='Z',camera=dict(eye=dict(x=1,y=-1,z=1))),font=dict(size=15,family='Times New Roman'),autosize=False,height=550,width=600,margin=dict(l=20,r=20,t=40,b=20),showlegend=False) # ,legend=dict(x=0.02,y=0.98),
fig.update_scenes(camera_projection_type='orthographic')
fig.show()

In [5]:
#| label: w4-main

df=pd.read_csv('../../data/2001/w4.csv')
doe_df=pd.read_csv('../../pyelastica/wiper_doe10p/data/doe_df.csv')[0:100]
prt=df.iloc[10:,2:].reset_index(drop=True).values
p0=df.iloc[5,2:].values.astype(float)   
p1=df.iloc[6,2:].values.astype(float)   
u0=df.iloc[8,2:].values.astype(float)*df.iloc[2,1]   
u1=df.iloc[9,2:].values.astype(float)*df.iloc[3,1]   
fig = go.Figure()
fig.add_trace(go.Scatter3d(x=prt[:,0],y=prt[:,1],z=prt[:,2],mode='lines',name='Prototype',line=dict(color='black',width=6)))
fig.add_trace(go.Scatter3d(x=[p0[0]],y=[p0[1]],z=[p0[2]],mode='markers',name='P0',marker=dict(size=5,color='red')))
fig.add_trace(go.Scatter3d(x=[p1[0]],y=[p1[1]],z=[p1[2]],mode='markers',name='P1',marker=dict(size=5,color='red')))
fig.add_trace(go.Cone(x=[p0[0]],y=[p0[1]],z=[p0[2]],u=[u0[0]],v=[u0[1]],w=[u0[2]],colorscale='RdPu',sizemode='absolute',sizeref=6,showscale=False,name='U@P0',showlegend=True))
fig.add_trace(go.Cone(x=[p1[0]],y=[p1[1]],z=[p1[2]],u=[u1[0]],v=[u1[1]],w=[u1[2]],colorscale='RdPu',sizemode='absolute',sizeref=6,showscale=False,name='U@P1',showlegend=True))
a=pd.DataFrame(prt).iloc[50].values

lst_err=[]
lst_ln=[]

angles = list(range(0,20))
cmap = cm.get_cmap('hsv', len(angles))   # choose any colormap

for idx, i in enumerate(angles):
    file_name=f'N={i+1}'
    spl3=pd.read_csv('../../pyelastica/wiper_doe10p/data/w4_iter_'+str(i+1)+'.csv')
    errb=length(pd.DataFrame([a,spl3.iloc[50]]).values).round(3)
    spl3=spl3.values
    color = mcolors.to_hex(cmap(idx))
    fig.add_trace(go.Scatter3d(x=spl3[:,0],y=spl3[:,1],z=spl3[:,2],mode='lines',name=file_name,line=dict(color=color,width=6),customdata=np.tile(np.array([i,doe_df["twist"].iloc[i],doe_df["youngs_modulus"].iloc[i]*1000,doe_df['base_radius'].iloc[i],doe_df['density'].iloc[i]*10e5,doe_df['coner_angle'].iloc[i],doe_df['length'].iloc[i]]), (spl3.shape[0],1)),hovertemplate=f"N= {i+1}<br>"+"twist= %{customdata[1]:.2f} deg<br>"+"E= %{customdata[2]:.2f} MPa<br>"+"r= %{customdata[3]:.2f} mm<br>"+"ro= %{customdata[4]:.2f} g/cm3<br>"+"cone angle= %{customdata[5]:.2f} deg<br>"+"length= %{customdata[6]:.2f} mm<br>"+"<extra></extra>"))
    ln_spl=length(spl3)-length(prt)
    lst_err.append(errb)
    lst_ln.append(ln_spl)
fig.update_layout(title='Simulated - W4 - DOE',scene=dict(xaxis_title='X',yaxis_title='Y',zaxis_title='Z',camera=dict(eye=dict(x=1,y=-1,z=1))),font=dict(size=15,family='Times New Roman'),autosize=False,height=550,width=600,margin=dict(l=20,r=20,t=40,b=20),showlegend=False) # ,legend=dict(x=0.02,y=0.98),
fig.update_scenes(camera_projection_type='orthographic')
fig.show()

In [6]:
#| label: w5-main

df=pd.read_csv('../../data/2001/w5.csv')
doe_df=pd.read_csv('../../pyelastica/wiper_doe10p/data/doe_df.csv')[0:100]
prt=df.iloc[10:,2:].reset_index(drop=True).values
p0=df.iloc[5,2:].values.astype(float)   
p1=df.iloc[6,2:].values.astype(float)   
u0=df.iloc[8,2:].values.astype(float)*df.iloc[2,1]   
u1=df.iloc[9,2:].values.astype(float)*df.iloc[3,1]   
fig = go.Figure()
fig.add_trace(go.Scatter3d(x=prt[:,0],y=prt[:,1],z=prt[:,2],mode='lines',name='Prototype',line=dict(color='black',width=6)))
fig.add_trace(go.Scatter3d(x=[p0[0]],y=[p0[1]],z=[p0[2]],mode='markers',name='P0',marker=dict(size=5,color='red')))
fig.add_trace(go.Scatter3d(x=[p1[0]],y=[p1[1]],z=[p1[2]],mode='markers',name='P1',marker=dict(size=5,color='red')))
fig.add_trace(go.Cone(x=[p0[0]],y=[p0[1]],z=[p0[2]],u=[u0[0]],v=[u0[1]],w=[u0[2]],colorscale='RdPu',sizemode='absolute',sizeref=6,showscale=False,name='U@P0',showlegend=True))
fig.add_trace(go.Cone(x=[p1[0]],y=[p1[1]],z=[p1[2]],u=[u1[0]],v=[u1[1]],w=[u1[2]],colorscale='RdPu',sizemode='absolute',sizeref=6,showscale=False,name='U@P1',showlegend=True))
a=pd.DataFrame(prt).iloc[50].values

lst_err=[]
lst_ln=[]

angles = list(range(0,20))
cmap = cm.get_cmap('hsv', len(angles))   # choose any colormap

for idx, i in enumerate(angles):
    file_name=f'N={i+1}'
    spl3=pd.read_csv('../../pyelastica/wiper_doe10p/data/w5_iter_'+str(i+1)+'.csv')
    errb=length(pd.DataFrame([a,spl3.iloc[50]]).values).round(3)
    spl3=spl3.values
    color = mcolors.to_hex(cmap(idx))
    fig.add_trace(go.Scatter3d(x=spl3[:,0],y=spl3[:,1],z=spl3[:,2],mode='lines',name=file_name,line=dict(color=color,width=6),customdata=np.tile(np.array([i,doe_df["twist"].iloc[i],doe_df["youngs_modulus"].iloc[i]*1000,doe_df['base_radius'].iloc[i],doe_df['density'].iloc[i]*10e5,doe_df['coner_angle'].iloc[i],doe_df['length'].iloc[i]]), (spl3.shape[0],1)),hovertemplate=f"N= {i+1}<br>"+"twist= %{customdata[1]:.2f} deg<br>"+"E= %{customdata[2]:.2f} MPa<br>"+"r= %{customdata[3]:.2f} mm<br>"+"ro= %{customdata[4]:.2f} g/cm3<br>"+"cone angle= %{customdata[5]:.2f} deg<br>"+"length= %{customdata[6]:.2f} mm<br>"+"<extra></extra>"))
    ln_spl=length(spl3)-length(prt)
    lst_err.append(errb)
    lst_ln.append(ln_spl)
fig.update_layout(title='Simulated - W5 - DOE',scene=dict(xaxis_title='X',yaxis_title='Y',zaxis_title='Z',camera=dict(eye=dict(x=1,y=-1,z=1))),font=dict(size=15,family='Times New Roman'),autosize=False,height=550,width=600,margin=dict(l=20,r=20,t=40,b=20),showlegend=False) # ,legend=dict(x=0.02,y=0.98),
fig.update_scenes(camera_projection_type='orthographic')
fig.show()

In [7]:
#| label: w1-doe-error

df=pd.read_csv('../../data/2001/w1.csv')
prt=df.iloc[10:,2:].reset_index(drop=True)
angles = list(range(0, 20))
cmap = cm.get_cmap('hsv', len(angles)) 
fig=tools.make_subplots(rows=2,cols=1,subplot_titles=("Deviation","Curvature"),vertical_spacing=0.1)

for idx, i in enumerate(angles):
    file_name=f'N={i+1}'
    spl3=pd.read_csv('../../pyelastica/wiper_doe10p/data/w1_iter_'+str(i+1)+'.csv') 
    histerr=[]
    color = mcolors.to_hex(cmap(idx))

    for _i in range(len(prt)):
        histerr.append(length(pd.DataFrame([pd.DataFrame(prt,columns=spl3.columns).iloc[_i],spl3.iloc[_i]]).values).round(3))
    fig1=go.Scatter(x=[_i for _i in range(1,101,1)],y=histerr,name=file_name,mode='lines',line=dict(color=color,width=3),customdata=np.tile(np.array([i,doe_df["twist"].iloc[i],doe_df["youngs_modulus"].iloc[i]*1000,doe_df['base_radius'].iloc[i],doe_df['density'].iloc[i]*10e5,doe_df['coner_angle'].iloc[i],doe_df['length'].iloc[i]]), (spl3.shape[0],1)),hovertemplate=f"N= {i+1}<br>"+"twist= %{customdata[1]:.2f} deg<br>"+"E= %{customdata[2]:.2f} MPa<br>"+"r= %{customdata[3]:.2f} mm<br>"+"ro= %{customdata[4]:.2f} g/cm3<br>"+"cone angle= %{customdata[5]:.2f} deg<br>"+"length= %{customdata[6]:.2f} mm<br>"+"<extra></extra>")
    fig2=plot_curvature(spl3,color,file_name)
    fig.append_trace(fig1, 1, 1)
    fig.append_trace(fig2, 2, 1)

fig.append_trace(plot_curvature(prt,'#000000','Prototype'), 2, 1)
fig.update_layout(font=dict(size=15,family='Times New Roman'),autosize=False,showlegend=False,height=550,width=410,margin=dict(l=20,r=20,t=20,b=20))
fig.update_xaxes(title_text="Node",row=2)
fig.update_yaxes(title_text="Deviation (mm)", row=1)
fig.update_yaxes(title_text="Curvature (rad/mm)", row=2)
iplot(fig)

In [8]:
#| label: w2-doe-error

df=pd.read_csv('../../data/2001/w2.csv')
prt=df.iloc[10:,2:].reset_index(drop=True)
angles = list(range(0, 20))
cmap = cm.get_cmap('hsv', len(angles)) 
fig=tools.make_subplots(rows=2,cols=1,subplot_titles=("Deviation","Curvature"),vertical_spacing=0.1)

for idx, i in enumerate(angles):
    file_name=f'N={i+1}'
    spl3=pd.read_csv('../../pyelastica/wiper_doe10p/data/w2_iter_'+str(i+1)+'.csv') 
    histerr=[]
    color = mcolors.to_hex(cmap(idx))

    for _i in range(len(prt)):
        histerr.append(length(pd.DataFrame([pd.DataFrame(prt,columns=spl3.columns).iloc[_i],spl3.iloc[_i]]).values).round(3))
    fig1=go.Scatter(x=[_i for _i in range(1,101,1)],y=histerr,name=file_name,mode='lines',line=dict(color=color,width=3),customdata=np.tile(np.array([i,doe_df["twist"].iloc[i],doe_df["youngs_modulus"].iloc[i]*1000,doe_df['base_radius'].iloc[i],doe_df['density'].iloc[i]*10e5,doe_df['coner_angle'].iloc[i],doe_df['length'].iloc[i]]), (spl3.shape[0],1)),hovertemplate=f"N= {i+1}<br>"+"twist= %{customdata[1]:.2f} deg<br>"+"E= %{customdata[2]:.2f} MPa<br>"+"r= %{customdata[3]:.2f} mm<br>"+"ro= %{customdata[4]:.2f} g/cm3<br>"+"cone angle= %{customdata[5]:.2f} deg<br>"+"length= %{customdata[6]:.2f} mm<br>"+"<extra></extra>")
    fig2=plot_curvature(spl3,color,file_name)
    fig.append_trace(fig1, 1, 1)
    fig.append_trace(fig2, 2, 1)

fig.append_trace(plot_curvature(prt,'#000000','Prototype'), 2, 1)
fig.update_layout(font=dict(size=15,family='Times New Roman'),autosize=False,showlegend=False,height=550,width=410,margin=dict(l=20,r=20,t=20,b=20))
fig.update_xaxes(title_text="Node",row=2)
fig.update_yaxes(title_text="Deviation (mm)", row=1)
fig.update_yaxes(title_text="Curvature (rad/mm)", row=2)
iplot(fig)

In [9]:
#| label: w3-doe-error

df=pd.read_csv('../../data/2001/w3.csv')
prt=df.iloc[10:,2:].reset_index(drop=True)
angles = list(range(0, 20))
cmap = cm.get_cmap('hsv', len(angles)) 
fig=tools.make_subplots(rows=2,cols=1,subplot_titles=("Deviation","Curvature"),vertical_spacing=0.1)

for idx, i in enumerate(angles):
    file_name=f'N={i+1}'
    spl3=pd.read_csv('../../pyelastica/wiper_doe10p/data/w3_iter_'+str(i+1)+'.csv') 
    histerr=[]
    color = mcolors.to_hex(cmap(idx))

    for _i in range(len(prt)):
        histerr.append(length(pd.DataFrame([pd.DataFrame(prt,columns=spl3.columns).iloc[_i],spl3.iloc[_i]]).values).round(3))
    fig1=go.Scatter(x=[_i for _i in range(1,101,1)],y=histerr,name=file_name,mode='lines',line=dict(color=color,width=3),customdata=np.tile(np.array([i,doe_df["twist"].iloc[i],doe_df["youngs_modulus"].iloc[i]*1000,doe_df['base_radius'].iloc[i],doe_df['density'].iloc[i]*10e5,doe_df['coner_angle'].iloc[i],doe_df['length'].iloc[i]]), (spl3.shape[0],1)),hovertemplate=f"N= {i+1}<br>"+"twist= %{customdata[1]:.2f} deg<br>"+"E= %{customdata[2]:.2f} MPa<br>"+"r= %{customdata[3]:.2f} mm<br>"+"ro= %{customdata[4]:.2f} g/cm3<br>"+"cone angle= %{customdata[5]:.2f} deg<br>"+"length= %{customdata[6]:.2f} mm<br>"+"<extra></extra>")
    fig2=plot_curvature(spl3,color,file_name)
    fig.append_trace(fig1, 1, 1)
    fig.append_trace(fig2, 2, 1)

fig.append_trace(plot_curvature(prt,'#000000','Prototype'), 2, 1)
fig.update_layout(font=dict(size=15,family='Times New Roman'),autosize=False,showlegend=False,height=550,width=410,margin=dict(l=20,r=20,t=20,b=20))
fig.update_xaxes(title_text="Node",row=2)
fig.update_yaxes(title_text="Deviation (mm)", row=1)
fig.update_yaxes(title_text="Curvature (rad/mm)", row=2)
iplot(fig)

In [10]:
#| label: w4-doe-error

df=pd.read_csv('../../data/2001/w4.csv')
prt=df.iloc[10:,2:].reset_index(drop=True)
angles = list(range(0, 20))
cmap = cm.get_cmap('hsv', len(angles)) 
fig=tools.make_subplots(rows=2,cols=1,subplot_titles=("Deviation","Curvature"),vertical_spacing=0.1)

for idx, i in enumerate(angles):
    file_name=f'N={i+1}'
    spl3=pd.read_csv('../../pyelastica/wiper_doe10p/data/w4_iter_'+str(i+1)+'.csv') 
    histerr=[]
    color = mcolors.to_hex(cmap(idx))

    for _i in range(len(prt)):
        histerr.append(length(pd.DataFrame([pd.DataFrame(prt,columns=spl3.columns).iloc[_i],spl3.iloc[_i]]).values).round(3))
    fig1=go.Scatter(x=[_i for _i in range(1,101,1)],y=histerr,name=file_name,mode='lines',line=dict(color=color,width=3),customdata=np.tile(np.array([i,doe_df["twist"].iloc[i],doe_df["youngs_modulus"].iloc[i]*1000,doe_df['base_radius'].iloc[i],doe_df['density'].iloc[i]*10e5,doe_df['coner_angle'].iloc[i],doe_df['length'].iloc[i]]), (spl3.shape[0],1)),hovertemplate=f"N= {i+1}<br>"+"twist= %{customdata[1]:.2f} deg<br>"+"E= %{customdata[2]:.2f} MPa<br>"+"r= %{customdata[3]:.2f} mm<br>"+"ro= %{customdata[4]:.2f} g/cm3<br>"+"cone angle= %{customdata[5]:.2f} deg<br>"+"length= %{customdata[6]:.2f} mm<br>"+"<extra></extra>")
    fig2=plot_curvature(spl3,color,file_name)
    fig.append_trace(fig1, 1, 1)
    fig.append_trace(fig2, 2, 1)

fig.append_trace(plot_curvature(prt,'#000000','Prototype'), 2, 1)
fig.update_layout(font=dict(size=15,family='Times New Roman'),autosize=False,showlegend=False,height=550,width=410,margin=dict(l=20,r=20,t=20,b=20))
fig.update_xaxes(title_text="Node",row=2)
fig.update_yaxes(title_text="Deviation (mm)", row=1)
fig.update_yaxes(title_text="Curvature (rad/mm)", row=2)
iplot(fig)

In [11]:
#| label: w5-doe-error

df=pd.read_csv('../../data/2001/w5.csv')
prt=df.iloc[10:,2:].reset_index(drop=True)
angles = list(range(0, 20))
cmap = cm.get_cmap('hsv', len(angles)) 
fig=tools.make_subplots(rows=2,cols=1,subplot_titles=("Deviation","Curvature"),vertical_spacing=0.1)

for idx, i in enumerate(angles):
    file_name=f'N={i+1}'
    spl3=pd.read_csv('../../pyelastica/wiper_doe10p/data/w5_iter_'+str(i+1)+'.csv') 
    histerr=[]
    color = mcolors.to_hex(cmap(idx))

    for _i in range(len(prt)):
        histerr.append(length(pd.DataFrame([pd.DataFrame(prt,columns=spl3.columns).iloc[_i],spl3.iloc[_i]]).values).round(3))
    fig1=go.Scatter(x=[_i for _i in range(1,101,1)],y=histerr,name=file_name,mode='lines',line=dict(color=color,width=3),customdata=np.tile(np.array([i,doe_df["twist"].iloc[i],doe_df["youngs_modulus"].iloc[i]*1000,doe_df['base_radius'].iloc[i],doe_df['density'].iloc[i]*10e5,doe_df['coner_angle'].iloc[i],doe_df['length'].iloc[i]]), (spl3.shape[0],1)),hovertemplate=f"N= {i+1}<br>"+"twist= %{customdata[1]:.2f} deg<br>"+"E= %{customdata[2]:.2f} MPa<br>"+"r= %{customdata[3]:.2f} mm<br>"+"ro= %{customdata[4]:.2f} g/cm3<br>"+"cone angle= %{customdata[5]:.2f} deg<br>"+"length= %{customdata[6]:.2f} mm<br>"+"<extra></extra>")
    fig2=plot_curvature(spl3,color,file_name)
    fig.append_trace(fig1, 1, 1)
    fig.append_trace(fig2, 2, 1)

fig.append_trace(plot_curvature(prt,'#000000','Prototype'), 2, 1)
fig.update_layout(font=dict(size=15,family='Times New Roman'),autosize=False,showlegend=False,height=550,width=410,margin=dict(l=20,r=20,t=20,b=20))
fig.update_xaxes(title_text="Node",row=2)
fig.update_yaxes(title_text="Deviation (mm)", row=1)
fig.update_yaxes(title_text="Curvature (rad/mm)", row=2)
iplot(fig)